# Signal redundancy — X vs X overlap

*Read-only informative artifact. Characterises how much the candidate signals
duplicate each other so a human can decide which to keep. No gate decisions,
no PROCEED/STOP verdict.*

## Questions an analyst asks of the signal set

- **Which signals are measuring the same thing?** If two signals rank players
  almost identically at a position, keeping both adds noise, not information.
- **Which overlaps are structural rather than empirical?** Some signals are
  *defined* in terms of others (`xgi = xg + xa`; `ict_index` aggregates
  `influence`, `creativity`, `threat`) — that redundancy is algebraic and
  always present, not a finding about this season.
- **Does one signal make another redundant for explaining the target?**
  (deferred — partial correlation against `total_points`.)

Pairwise overlap below is **season-pooled** rank correlation. Whether two
signals diverge in specific weeks is deferred to the `temporal/` layer.

## Setup

Load the mart, restrict to the **whole season** (GW 1 to the latest completed
GW) and the **participation** population (`minutes > 0`), build position
cohorts, import the redundancy kernel, and define the signal set.

This is a *descriptive characterisation* notebook, so it uses the full season —
no early-GW lower bound (that GW-6 cut in the older EDA-1 record was a
*predictive-evaluation* choice, not relevant here).

The population is everyone who **actually featured**: available players with
`minutes > 0`. This is a **participation** filter (the player appeared), **not
a performance gate**. `minutes` can be NULL for some rows; `minutes > 0`
naturally excludes those. The 60-minute performance-boundary question is
**deferred to the `population/` layer** — the layer meant to study and justify
that boundary — and is deliberately not baked in here.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart, run as run_pipeline
from dal.exceptions import MartNotBuiltError, MartSchemaError
from research.kernels.descriptive.distribution import (
    compute_distribution_stats,
    compare_cohorts,
)
from research.kernels.diagnostic.redundancy import (
    compute_pairwise_rho,
    identify_redundant_pairs,
    DEFAULT_REDUNDANCY_THRESHOLD,
)

try:
    _result = load_mart()
except (MartNotBuiltError, MartSchemaError) as _e:
    print(f"Rebuilding mart ({type(_e).__name__})...")
    run_pipeline(force=True)
    _result = load_mart()

# Descriptive characterisation uses the WHOLE season: GW 1 to the latest
# completed GW. No early-GW lower bound (that was a predictive-evaluation
# choice in the older EDA-1 record, not relevant here).
STUDY_GW_MIN = 1
STUDY_GW_MAX = _result.data_cutoff_gw

# Analytical population: PARTICIPATION filter, not a performance gate.
# Available players who actually featured -> minutes > 0. `minutes` can be NULL
# for some rows; minutes > 0 naturally excludes NULLs (NULL comparisons are
# False), stated explicitly below. The 60-minute performance boundary is NOT
# imposed here -- that question is deferred to the population/ layer.
mart = _result.mart
df = mart[mart["gw"].between(STUDY_GW_MIN, STUDY_GW_MAX)].copy()
df = df[df["minutes"].notna() & (df["minutes"] > 0)].copy()

POSITIONS = ["GK", "DEF", "MID", "FWD"]
cohorts = {pos: df[df.position == pos] for pos in POSITIONS}

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.3f}".format)

print(f"Study range: GW {STUDY_GW_MIN} - GW {STUDY_GW_MAX} (whole season, from mart data_cutoff_gw)")
print(f"Population: minutes > 0 (participation, not a performance gate), n = {len(df):,} player-gameweeks")
for pos in POSITIONS:
    print(f"  {pos}: {len(cohorts[pos]):>6,}")

# Same candidate signal set as signal.ipynb.
SIGNALS = [
    "xg", "xa", "xgi", "xgc",
    "ict_index", "influence", "creativity", "threat",
    "bps", "bonus",
]
SIGNALS = [s for s in SIGNALS if s in df.columns]
print("\nSignals under test:", SIGNALS)


## (a) Pairwise Spearman rho by position

**What we measure** — for each position, the symmetric matrix of pairwise
**Spearman rho** across all signal pairs via `compute_pairwise_rho`, then the
pairs whose `|rho|` meets the kernel's redundancy threshold via
`identify_redundant_pairs`. (The kernel skips cells with fewer than 30
observations or a constant signal — those appear as NaN.)

**What it means** — high rank correlation at a position means the two signals
order that position's players almost identically; for ranking purposes they
are near-substitutes there. The flagged pairs are the first place to look when
trimming a redundant signal set.

**What it doesn't mean** — `|rho| >= threshold` is an **operational heuristic**
(`DEFAULT_REDUNDANCY_THRESHOLD`, no statistical derivation), not a proof of
duplication. Rank correlation ignores magnitude, so two signals can be
rank-redundant yet differ in scale. These are **season-pooled** correlations;
two signals that usually agree can diverge in specific weeks — that is deferred
to the `temporal/` layer. High rho does not say which member of a pair to keep.

**Guiding question** — *Which signals are measuring the same thing at each
position?*

In [ ]:
threshold = DEFAULT_REDUNDANCY_THRESHOLD
print(f"Redundancy threshold |rho| >= {threshold} (operational heuristic from kernel)\n")
redundant_by_pos = {}
for pos in POSITIONS:
    rho = compute_pairwise_rho(df, SIGNALS, pos)
    pairs = identify_redundant_pairs(rho, threshold=threshold)
    redundant_by_pos[pos] = pairs
    print(f"=== {pos} === redundant pairs (|rho| >= {threshold}):")
    if pairs:
        for a, b in pairs:
            print(f"    {a:>12} ~ {b:<12}  rho = {rho.loc[a, b]:.3f}")
    else:
        print("    (none)")
    print()


In [ ]:
# Heatmaps reveal the correlation block-structure a flat pair-list hides.
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, pos in zip(axes, POSITIONS):
    rho = compute_pairwise_rho(df, SIGNALS, pos)
    im = ax.imshow(rho.values.astype(float), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(SIGNALS)))
    ax.set_xticklabels(SIGNALS, rotation=90, fontsize=7)
    ax.set_yticks(range(len(SIGNALS)))
    ax.set_yticklabels(SIGNALS, fontsize=7)
    ax.set_title(pos)
fig.colorbar(im, ax=axes, label="Spearman rho", shrink=0.7)
fig.suptitle("Pairwise Spearman rho by position (season-pooled)", y=1.02)
plt.show()


## (b) Algebraic-decomposition flags

**What we measure** — pairs/groups whose redundancy is **structural by
definition**, flagged from the signal names (not from data):

- `xgi == xg + xa` — expected goal involvements is the sum of its parts;
- `ict_index` aggregates `influence`, `creativity`, `threat`.

We confirm the `xgi = xg + xa` identity empirically (max absolute residual)
and label the `ict_index` component group.

**What it means** — these overlaps are mechanical and **always present**,
independent of season. A high rho between `xgi` and `xg` in section (a) is not
a discovery — it is arithmetic. Keeping both a composite and its components in
a ranking model double-counts the same underlying contribution.

**What it doesn't mean** — flagging the algebra does not say which
representation to keep (composite vs components); that is a modelling choice
about whether the components carry separable signal, not a redundancy fact. The
residual check validates the identity in this mart; it is not a test of
predictive value.

**Guiding question** — *Which overlaps are structural rather than empirical?*

In [ ]:
# Confirm xgi = xg + xa as a definitional identity in this mart.
# Residual is bounded by the mart's 2-decimal storage rounding, not a real gap.
if {"xgi", "xg", "xa"}.issubset(df.columns):
    resid = (df["xgi"].astype(float) - (df["xg"].astype(float) + df["xa"].astype(float))).abs()
    print(f"xgi vs (xg + xa):  max |residual| = {resid.max():.6f}  "
          f"(<= storage rounding -> algebraic identity holds)")

print("\nAlgebraic-decomposition groups (structural, season-independent):")
algebraic_groups = {
    "xgi = xg + xa": ["xgi", "xg", "xa"],
    "ict_index = influence + creativity + threat (composite)":
        ["ict_index", "influence", "creativity", "threat"],
}
for label, members in algebraic_groups.items():
    present = [m for m in members if m in df.columns]
    print(f"  - {label}")
    print(f"      members present: {present}")


## DEFERRED — pending kernel: X-Y partial-correlation redundancy

This section will test whether, **for explaining the target**, one signal makes
another redundant — i.e. whether signal B's correlation with `total_points`
survives after partialling out signal A (partial Spearman / semi-partial rho,
per position). Two signals can be only moderately correlated with each other
yet one fully subsumes the other's explanatory content for Y; pairwise X-vs-X
rho in section (a) cannot detect that.

It requires a partial-correlation kernel (rank-based, with the same
min-observations and NaN-handling guarantees as `compute_pairwise_rho`) that is
not yet built in `research.kernels.diagnostic`. No code is run here until that
kernel exists — a target-aware redundancy computation belongs in a kernel, not
inline.

## What the redundancy picture looks like

Plain-language summary (not a verdict):

- The strongest, most consistent overlaps are the **algebraic** ones: `xgi`
  with `xg`/`xa`, and `ict_index` with `threat`/`creativity`/`influence`.
  These appear at every position by construction and are confirmed by the
  `xgi = xg + xa` residual being effectively zero.
- Empirical (non-algebraic) overlap is **position-dependent**: attacking
  signals cluster together for MID/FWD, while for GK/DEF many attacking pairs
  fall below the threshold or are NaN (too few non-constant observations).
- No structural-vs-empirical distinction is auto-resolved here; the flags point
  a human at the candidates.

All pairwise figures are **whole-season**, season-pooled, over the
**participation** population (`minutes > 0` — the player appeared — not a
performance gate; the 60-minute boundary is deferred to the `population/`
layer). Week-to-week divergence and the target-aware (partial-correlation) view
are deferred above and to the `temporal/` layer.